# NBody-rs: benchmark plots

Loads the files written by `run_bench.sh` into `data/` and plots the first three experiments of `REPORT.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import ticker

DATA = Path("data")  # relative to the report directory
MARK = "--- OUTPUT ---"

# Settings of run_bench.sh: other files in data/ (older runs) are ignored.
CUDA_BLOCKS = (1, 16, 64, 256, 1024)
CUDA_THREADS = (1, 16, 64, 256)
ITERS = 1000  # iterations of experiments 1 and 3
ITERS_THETA = 50  # iterations of experiment 2
THETA_GRID = 30  # grid length used by the theta experiment
STEPS_GRID = 50  # grid length used by the per-step experiment

# CUDA lines: color = number of blocks, line style and marker = threads per block
# (red is reserved for the CPU).
BLOCK_COLORS = dict(zip(CUDA_BLOCKS, ["tab:blue", "tab:orange", "tab:green", "tab:purple", "tab:brown"]))
THREAD_STYLES = dict(zip(CUDA_THREADS, [(":", "v"), ("-.", "s"), ("--", "^"), ("-", "o")]))


def cuda_style(cfg):
    """Plot style of a `cuda:<blocks>x<threads>` backend."""
    blocks, threads = map(int, cfg[5:].split("x"))
    linestyle, marker = THREAD_STYLES[threads]
    return dict(color=BLOCK_COLORS[blocks], linestyle=linestyle, marker=marker, markersize=3, linewidth=1.3)


def log_axis(axis):
    """Labels at 1, 2 and 5 of every decade of a log axis (call after set_*scale)."""
    axis.set_major_locator(ticker.LogLocator(base=10, subs=(1, 2, 5), numticks=30))
    axis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    axis.set_minor_locator(ticker.LogLocator(base=10, subs=range(1, 10), numticks=30))
    axis.set_minor_formatter(ticker.NullFormatter())


def linear_axis(axis, n=12):
    """About n labels on a linear axis."""
    axis.set_major_locator(ticker.MaxNLocator(n))

## Parsing

Each file starts with `# params: <bench arguments>`, followed by the CSV (`Step,Time ns`) enclosed between two `--- OUTPUT ---` lines. Files without both markers (incomplete runs) are ignored.

In [ ]:
def parse(path):
    lines = path.read_text().splitlines()
    if not lines or not lines[0].startswith("# params:"):
        return None
    marks = [i for i, l in enumerate(lines) if l.strip() == MARK]
    if len(marks) < 2:
        return None

    p = lines[0].split()
    arg = lambda flag: p[p.index(flag) + 1]
    scenario, backend = arg("-s"), arg("-b")

    length = int(scenario.split(":")[1]) if scenario.startswith("grid:") else np.nan
    blocks = threads = np.nan
    if backend.startswith("cuda:"):
        blocks, threads = map(int, backend[5:].split("x"))

    # lines[marks[0] + 1] is the CSV header
    rows = lines[marks[0] + 2 : marks[1]]
    times_ms = np.array([int(r.split(",")[1]) for r in rows]) / 1e6

    return dict(
        scenario=scenario.split(":")[0],
        length=length,
        bodies=(2 * length) ** 2,  # grid:<L> spawns (2L)^2 bodies
        theta=float(arg("-T")),
        iters=int(arg("-n")),
        backend=backend,
        kind="cuda" if backend.startswith("cuda:") else "cpu",
        blocks=blocks,
        threads=threads,
        mean_ms=times_ms.mean(),
        times_ms=times_ms,
    )


runs = pd.DataFrame([r for r in map(parse, sorted(DATA.glob("*.txt"))) if r])
print(f"{len(runs)} complete runs loaded")

# keep the CPU and the CUDA configurations used by run_bench.sh
runs = runs[(runs.kind == "cpu") | (runs.blocks.isin(CUDA_BLOCKS) & runs.threads.isin(CUDA_THREADS))]
print(f"{len(runs)} runs after filtering the CUDA configurations")
runs.drop(columns="times_ms").head()

## Experiment 1: mean iteration time vs number of bodies (grid, theta = 0.5)

One line per CUDA configuration (blocks x threads), plus the CPU.

In [ ]:
e1 = runs[(runs.scenario == "grid") & (runs.theta == 0.5) & (runs.iters == ITERS)]
cpu = e1[e1.kind == "cpu"].sort_values("bodies")
cuda = e1[e1.kind == "cuda"]
best = cuda.loc[cuda.groupby("bodies").mean_ms.idxmin()].sort_values("bodies")

# mean iteration time normalized by the number of bodies [ns / body]
per_body = lambda d: d.mean_ms * 1e6 / d.bodies

fig, ax = plt.subplots(figsize=(11, 6))
configs = sorted(cuda.backend.unique(), key=lambda b: tuple(map(int, b[5:].split("x"))))
for cfg in configs:
    g = cuda[cuda.backend == cfg].sort_values("bodies")
    ax.plot(g.bodies, per_body(g), label=f"CUDA {cfg[5:]}", **cuda_style(cfg))
ax.plot(cpu.bodies, per_body(cpu), "o-", color="tab:red", linewidth=2.5, label="CPU")
ax.set_xscale("log")
ax.set_yscale("log")
log_axis(ax.xaxis)
log_axis(ax.yaxis)
ax.set_xlabel("bodies")
ax.set_ylabel("mean iteration time per body [ns]")
ax.set_title("Grid scenario, theta = 0.5 (CUDA legend: blocks x threads)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), ncol=1, fontsize=8)
fig.savefig("images/plots/exp1_time_per_body.svg", bbox_inches="tight")
plt.show()

Speedup of the best CUDA configuration over the CPU.

In [ ]:
sp = cpu.merge(best, on="bodies", suffixes=("_cpu", "_cuda"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sp.bodies, sp.mean_ms_cpu / sp.mean_ms_cuda, "o-")
ax.axhline(1, color="gray", linestyle="--")
ax.set_xscale("log")
log_axis(ax.xaxis)
linear_axis(ax.yaxis)
ax.set_xlabel("bodies")
ax.set_ylabel("speedup (CPU time / CUDA time)")
ax.grid(True, which="both", alpha=0.3)
fig.savefig("images/plots/exp1_speedup.svg", bbox_inches="tight")
plt.show()

## Experiment 2: mean iteration time vs theta (fixed grid size)

In [ ]:
e2 = runs[(runs.scenario == "grid") & (runs.length == THETA_GRID) & (runs.iters == ITERS_THETA)]
cpu2 = e2[e2.kind == "cpu"].sort_values("theta")
cuda2 = e2[e2.kind == "cuda"]

# mean iteration time normalized by the number of bodies [ns / body]
per_body2 = lambda d: d.mean_ms * 1e6 / d.bodies

fig, ax = plt.subplots(figsize=(11, 6))
configs2 = sorted(cuda2.backend.unique(), key=lambda b: tuple(map(int, b[5:].split("x"))))
for cfg in configs2:
    g = cuda2[cuda2.backend == cfg].sort_values("theta")
    ax.plot(g.theta, per_body2(g), label=f"CUDA {cfg[5:]}", **cuda_style(cfg))
ax.plot(cpu2.theta, per_body2(cpu2), "o-", color="tab:red", linewidth=2.5, label="CPU")
ax.set_yscale("log")
log_axis(ax.yaxis)
ax.set_xticks(sorted(e2.theta.unique()))  # one label per measured theta
ax.set_xlabel("theta")
ax.set_ylabel("mean iteration time per body [ns]")
ax.set_title(f"Grid scenario, {(2 * THETA_GRID) ** 2} bodies (CUDA legend: blocks x threads)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), ncol=1, fontsize=8)
fig.savefig("images/plots/exp2_theta.svg", bbox_inches="tight")
plt.show()

## Experiment 3: time of each iteration (grid and orbit scenarios, theta = 0.5)

For CUDA, the configuration with the lowest mean iteration time in each scenario is shown.

In [ ]:
def gap_scale(regions):
    """Piecewise-linear scale. Each region (lo, hi, factor) rescales the range [lo, hi] ms
    to `factor` times its normal height (<1 squeezes, >1 expands); everything else is linear.
    Regions must not overlap."""
    xs, zs = [], []
    z = prev = None
    for lo, hi, factor in sorted(regions):
        z = lo if z is None else z + (lo - prev)
        xs.append(lo)
        zs.append(z)
        z += (hi - lo) * factor
        xs.append(hi)
        zs.append(z)
        prev = hi

    def fwd(y):
        y = np.asarray(y, dtype=float)
        return np.interp(y, xs, zs) + np.where(y < xs[0], y - xs[0], 0) + np.where(y > xs[-1], y - xs[-1], 0)

    def inv(z):
        z = np.asarray(z, dtype=float)
        return np.interp(z, zs, xs) + np.where(z < zs[0], z - zs[0], 0) + np.where(z > zs[-1], z - zs[-1], 0)

    return fwd, inv


# (title, rows of the scenario, gaps, between) -- per scenario:
#   gaps    = list of squeezed/expanded ranges (GAP_LO, GAP_HI, GAP_FACTOR), shaded in gray
#   between = height factor of the ranges between consecutive gaps (>1 makes them bigger)
scenarios = [
    (
        f"grid ({(2 * STEPS_GRID) ** 2} bodies)",
        (runs.scenario == "grid") & (runs.length == STEPS_GRID),
        [(0, 3.5, 0.1), (5, 91, 0.01)],
        5,
    ),
    (
        "orbit",
        runs.scenario == "orbit",
        [(0, 2.5, 0.1), (4, 36, 0.01)],
        5,
    ),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, (title, mask, gaps, between) in zip(axes, scenarios):
    gaps = sorted(gaps)
    e3 = runs[mask & (runs.theta == 0.5) & (runs.iters == ITERS)]
    cpu3 = e3[e3.kind == "cpu"]
    cuda3 = e3[e3.kind == "cuda"]
    series = []
    if len(cpu3):
        t = cpu3.iloc[0].times_ms
        series.append(t)
        ax.plot(t, color="tab:red", linewidth=0.8, label="CPU")
    if len(cuda3):
        b = cuda3.loc[cuda3.mean_ms.idxmin()]
        series.append(b.times_ms)
        ax.plot(b.times_ms, color="tab:blue", linewidth=0.8, label=f"CUDA {b.backend[5:]}")

    # scale = the gaps plus the (enlarged) ranges between them
    betweens = [(hi, nxt_lo, between) for (_, hi, _), (nxt_lo, _, _) in zip(gaps, gaps[1:]) if nxt_lo > hi]
    ax.set_yscale("function", functions=gap_scale(gaps + betweens))
    ymax = max(max((s.max() for s in series), default=0), gaps[-1][1] + 1)

    # ticks: gap edges, plus ticks inside the linear ranges between the gaps
    starts = [0] + [hi for _, hi, _ in gaps]
    ends = [lo for lo, _, _ in gaps] + [ymax]
    ticks = {edge for lo, hi, _ in gaps for edge in (lo, hi)}
    for a, b_ in zip(starts, ends):
        if b_ > a:
            ticks.update(ticker.MaxNLocator(6, steps=[1, 2, 5, 10]).tick_values(a, b_))
    ticks = sorted(t for t in ticks if 0 <= t <= ymax * 1.05)
    ax.yaxis.set_major_locator(ticker.FixedLocator(ticks))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.yaxis.set_minor_locator(ticker.NullLocator())
    for lo, hi, factor in gaps:
        ax.axhspan(lo, hi, color="gray", alpha=0.15)
    ax.set_ylim(0, ymax * 1.05)

    linear_axis(ax.xaxis)
    ax.set_xlabel("iteration")
    ax.set_ylabel("iteration time [ms]")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
fig.savefig("images/plots/exp3_iterations.svg", bbox_inches="tight")
plt.show()